# 🎵 Cantautore Digitale — Studio Colab
### GPU Gratis | Genera canzoni | Gestisci album | Scarica stems

**Come usare:**
1. Clicca `Runtime → Change runtime type → T4 GPU`
2. Premi `Ctrl+F9` (Run All) oppure esegui cella per cella
3. Inserisci la tua API Key di Google quando richiesto
4. Genera le tue canzoni!

**Costo:** ~€0.10 per canzone (solo API Google, GPU gratis da Colab)

In [ ]:
#@title 🔧 **STEP 1: Setup completo** (esegui UNA VOLTA — ~5 min)
#@markdown Installa tutto: repo, dipendenze, modelli. Aspetta che finisca.

import subprocess, os, sys

# Verifica GPU
gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                           capture_output=True, text=True)
if gpu_check.returncode == 0:
    print(f"✅ GPU trovata: {gpu_check.stdout.strip()}")
else:
    print("❌ NESSUNA GPU! Vai su Runtime → Change runtime type → T4 GPU")
    raise SystemExit

# Clona repo
if not os.path.exists("/content/cantautore"):
    print("📥 Clono il repo...")
    subprocess.run(["git", "clone", "https://github.com/mcldigital888-ship-it/cantautore-per-sigle.git",
                     "/content/cantautore"], check=True)
    print("✅ Repo clonato")
else:
    # Aggiorna
    subprocess.run(["git", "pull"], cwd="/content/cantautore", check=True)
    print("✅ Repo aggiornato")

os.chdir("/content/cantautore")
sys.path.insert(0, "/content/cantautore")

# Installa dipendenze
print("\n📦 Installazione dipendenze...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "google-genai", "demucs", "soundfile", "librosa",
                "rich", "python-dotenv", "pydub", "flask", "scipy"],
               check=True)
print("✅ Dipendenze installate")

# Verifica torch + CUDA
import torch
print(f"\n✅ PyTorch {torch.__version__} — CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Crea cartelle
for d in ["output", "temp", "voice", "voices", "models"]:
    os.makedirs(f"/content/cantautore/{d}", exist_ok=True)

print("\n🎉 Setup completato! Vai allo Step 2.")

In [ ]:
#@title 🔑 **STEP 2: API Key Google** (obbligatorio)
#@markdown Vai su https://aistudio.google.com/apikey → Crea chiave → Incollala qui

import os
from getpass import getpass

api_key = getpass("🔑 Incolla la tua Gemini API Key: ")
os.environ["GEMINI_API_KEY"] = api_key

# Salva in .env per i moduli
with open("/content/cantautore/.env", "w") as f:
    f.write(f"GEMINI_API_KEY={api_key}\n")

# Verifica
from google import genai
client = genai.Client(api_key=api_key)
test = client.models.generate_content(model="gemini-2.5-flash", contents="Rispondi solo: OK")
if test.text:
    print(f"✅ API Key funzionante!")
else:
    print("❌ Chiave non valida, riprova")

---
## 🎤 Voce (opzionale ma consigliato)
Carica la tua registrazione per usare la TUA voce nelle canzoni.
Se salti questo step, il sistema usa la voce generata da Lyria.

In [ ]:
#@title 🎤 **STEP 3: Carica la tua voce** (opzionale)
#@markdown Carica un file .wav della tua voce (30-60 secondi di canto)

import shutil, json
from google.colab import files
from pathlib import Path
from datetime import datetime

VOICES_DIR = Path("/content/cantautore/voices")
VOICE_DIR = Path("/content/cantautore/voice")

#@markdown Nome del profilo voce:
voice_name = "mia_voce" #@param {type:"string"}

print("📤 Seleziona il file .wav della tua voce...")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]

    # Salva nella cartella voices
    voice_dir = VOICES_DIR / voice_name
    voice_dir.mkdir(parents=True, exist_ok=True)
    dest = voice_dir / "reference.wav"
    shutil.move(filename, str(dest))

    # Aggiorna config
    config_file = VOICES_DIR / "voices_config.json"
    config = {"default_voice": voice_name, "profiles": {}}
    if config_file.exists():
        config = json.loads(config_file.read_text())
    config["profiles"][voice_name] = {
        "added": datetime.now().isoformat(),
        "source_file": filename,
    }
    config["default_voice"] = voice_name
    config_file.write_text(json.dumps(config, indent=2))

    # Copia in legacy per compatibilita
    VOICE_DIR.mkdir(exist_ok=True)
    shutil.copy2(str(dest), str(VOICE_DIR / "artist_voice.wav"))

    # Info
    import soundfile as sf
    info = sf.info(str(dest))
    print(f"\n✅ Voce '{voice_name}' caricata!")
    print(f"   Durata: {info.duration:.1f}s")
    print(f"   Sample rate: {info.samplerate}Hz")
    print(f"   Canali: {info.channels}")
    print(f"\n   Usa con: --voce {voice_name}")
else:
    print("⏭️ Nessuna voce caricata — useremo la voce AI di Lyria")

---
## 🎵 Genera Canzoni
Scegli il tema, la voce, e premi play. Puoi generare una canzone singola o un album intero.

In [ ]:
#@title 🎵 **Genera UNA canzone**
#@markdown Descrivi il tema e premi play!

import os, sys, time
os.chdir("/content/cantautore")
sys.path.insert(0, "/content/cantautore")

#@markdown ---
#@markdown ### Parametri canzone
tema = "nostalgia di un viaggio in treno verso Roma di notte" #@param {type:"string"}
titolo = "" #@param {type:"string"}
voce = "mia_voce" #@param {type:"string"}
export_stems = True #@param {type:"boolean"}

#@markdown ---

print("=" * 60)
print(f"🎵 GENERAZIONE CANZONE")
print(f"   Tema: {tema}")
print(f"   Voce: {voce or 'default'}")
print(f"   Stems: {'Si' if export_stems else 'No'}")
print("=" * 60)

start = time.time()

# Importa e genera
from cantautore import genera_canzone

result = genera_canzone(
    tema=tema,
    titolo=titolo if titolo else None,
    voice_name=voce if voce else None,
    export_stems=export_stems,
)

elapsed = time.time() - start
print(f"\n⏱️ Tempo totale: {int(elapsed // 60)}m {int(elapsed % 60)}s")
print(f"📁 File: {result}")

In [ ]:
#@title 🎧 **Ascolta l'ultima canzone generata**
#@markdown Ascolta direttamente nel browser

import IPython.display as ipd
from pathlib import Path

output_dir = Path("/content/cantautore/output")
wavs = sorted(output_dir.glob("*.wav"), key=lambda f: f.stat().st_mtime, reverse=True)
wavs = [w for w in wavs if "_kit" not in str(w.parent.name)]

if wavs:
    latest = wavs[0]
    print(f"🎵 {latest.name}")

    # Mostra testo se disponibile
    testo_file = latest.parent / f"{latest.stem}_testo.json"
    if testo_file.exists():
        import json
        data = json.loads(testo_file.read_text())
        print(f"\n📝 {data.get('titolo', '')} — {data.get('mood', '')} {data.get('bpm', '')}BPM\n")
        print(data.get("testo", "")[:500])
        print("\n" + "─" * 40)

    import soundfile as sf
    info = sf.info(str(latest))
    print(f"⏱️ Durata: {int(info.duration // 60)}:{int(info.duration % 60):02d}")
    print(f"📦 Dimensione: {latest.stat().st_size / (1024*1024):.1f} MB\n")

    ipd.display(ipd.Audio(str(latest)))
else:
    print("❌ Nessuna canzone trovata — genera prima una canzone")

In [ ]:
#@title 📀 **Genera ALBUM** (10 canzoni)
#@markdown Album completo con temi variati. ~€1 totale, ~2 ore.

import os, sys, time, json
os.chdir("/content/cantautore")

#@markdown ---
#@markdown ### Parametri album
nome_album = "Impronte Digitali" #@param {type:"string"}
voce_album = "mia_voce" #@param {type:"string"}
export_stems_album = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### Temi (modifica come vuoi, uno per riga)

temi_raw = """una donna che ti ha lasciato e che rivedi dopo 10 anni in un supermercato
l'ultima sigaretta fumata insieme a qualcuno che non vedrai mai piu sul balcone alle 5 di mattina
un uomo che parla con la segreteria telefonica della ex perche e l'unico modo di sentire la sua voce
l'Italia che va a rotoli ma tutti ballano, mood ALLEGRO e ironico da ballare
i social media che ci stanno mangiando il cervello ma non riusciamo a smettere, mood UPBEAT
il lavoro che ti ruba la vita 40 anni di mutuo per un appartamento che odii, raccontato come fosse una festa
un uomo che a 40 anni si accorge che ha passato la vita a inseguire cose sbagliate
la paura di morire non e morire e morire senza aver mai vissuto davvero
la storia di un barista che conosce i segreti di tutti nel quartiere ma nessuno conosce i suoi
Roma di notte i sampietrini bagnati il rumore del motorino in lontananza""" #@param {type:"string"}

temi = [t.strip() for t in temi_raw.strip().split("\n") if t.strip()]

print("=" * 60)
print(f"📀 ALBUM: {nome_album}")
print(f"   {len(temi)} tracce | Voce: {voce_album or 'default'}")
print(f"   Stems: {'Si' if export_stems_album else 'No'}")
print(f"   Costo stimato: ~€{len(temi) * 0.10:.2f}")
print("=" * 60)

from cantautore import genera_canzone

results = []
start = time.time()

for i, tema in enumerate(temi):
    print(f"\n{'='*60}")
    print(f"🎵 Traccia {i+1}/{len(temi)}")
    print(f"   Tema: {tema[:70]}...")
    print(f"{'='*60}")

    try:
        path = genera_canzone(
            tema=tema,
            voice_name=voce_album if voce_album else None,
            export_stems=export_stems_album,
        )
        results.append({"traccia": i+1, "file": str(path), "tema": tema, "ok": True})
        print(f"✅ Traccia {i+1} completata: {path.name}")
    except Exception as e:
        results.append({"traccia": i+1, "tema": tema, "ok": False, "errore": str(e)})
        print(f"❌ Traccia {i+1} fallita: {e}")

    time.sleep(3)  # Rate limit

elapsed = time.time() - start
ok = sum(1 for r in results if r["ok"])

print(f"\n{'='*60}")
print(f"📀 ALBUM COMPLETATO")
print(f"   {ok}/{len(temi)} tracce riuscite")
print(f"   ⏱️ Tempo: {int(elapsed // 60)}m {int(elapsed % 60)}s")
print(f"   💰 Costo: ~€{ok * 0.10:.2f}")
print(f"{'='*60}")

In [ ]:
#@title 🎧 **Ascolta TUTTE le canzoni**
#@markdown Player per tutte le canzoni generate

import IPython.display as ipd
import soundfile as sf
from pathlib import Path

output_dir = Path("/content/cantautore/output")
wavs = sorted(output_dir.glob("*.wav"), key=lambda f: f.stat().st_mtime, reverse=True)
wavs = [w for w in wavs if "_kit" not in str(w.parent.name)]

if not wavs:
    print("❌ Nessuna canzone — genera prima!")
else:
    print(f"🎵 {len(wavs)} canzoni trovate\n")
    for i, wav in enumerate(wavs):
        info = sf.info(str(wav))
        dur = f"{int(info.duration // 60)}:{int(info.duration % 60):02d}"
        size = wav.stat().st_size / (1024*1024)

        # Cerca titolo dal testo
        testo_f = wav.parent / f"{wav.stem}_testo.json"
        title = wav.stem
        mood = ""
        if testo_f.exists():
            import json
            d = json.loads(testo_f.read_text())
            title = d.get("titolo", wav.stem)
            mood = f" | {d.get('mood', '')} {d.get('bpm', '')}BPM"

        print(f"{'─'*50}")
        print(f"#{i+1} — {title} [{dur}] {size:.1f}MB{mood}")
        ipd.display(ipd.Audio(str(wav)))

In [ ]:
#@title 📥 **Scarica TUTTO** (canzoni + stems in ZIP)
#@markdown Scarica tutte le canzoni e gli stems in un unico file ZIP

import zipfile, os
from pathlib import Path
from google.colab import files

output_dir = Path("/content/cantautore/output")
zip_path = "/content/cantautore_output.zip"

# Conta file
all_files = list(output_dir.rglob("*"))
all_files = [f for f in all_files if f.is_file()]

if not all_files:
    print("❌ Nessun file da scaricare")
else:
    print(f"📦 Compressione {len(all_files)} file...")

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in all_files:
            arcname = f.relative_to(output_dir)
            zf.write(str(f), str(arcname))

    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"✅ ZIP creato: {size_mb:.1f} MB")
    print("📥 Download in corso...")
    files.download(zip_path)

---
## 🎛️ Strumenti Pro
Importa basi esterne, separa stems, converti formati

In [ ]:
#@title 🎹 **Importa una BASE esterna e cantaci sopra**
#@markdown Carica un beat/strumentale e il sistema genera testo + canta sopra

import os, sys, shutil
os.chdir("/content/cantautore")

from google.colab import files
from pathlib import Path

#@markdown ---
#@markdown ### Parametri
tema_base = "amore perduto in una notte a Milano" #@param {type:"string"}
voce_base = "mia_voce" #@param {type:"string"}
solo_testo = False #@param {type:"boolean"}

print("📤 Carica il file della base/beat (.wav)...")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    base_path = f"/content/cantautore/temp/{filename}"
    shutil.move(filename, base_path)

    from import_base import pipeline_importa_base

    result = pipeline_importa_base(
        base_path=base_path,
        tema=tema_base if tema_base else None,
        voice_name=voce_base if voce_base else None,
        solo_testo=solo_testo,
    )

    if result and not solo_testo:
        # Player
        import IPython.display as ipd
        mix_file = result / "mix_finale.wav"
        if mix_file.exists():
            print(f"\n🎵 Ascolta il risultato:")
            ipd.display(ipd.Audio(str(mix_file)))
else:
    print("❌ Nessun file caricato")

In [ ]:
#@title 🔪 **Separa qualsiasi canzone in stems** (per DAW)
#@markdown Carica un audio e ottieni: drums, bass, vocals, other, instrumental

import os, sys
os.chdir("/content/cantautore")

from google.colab import files
from pathlib import Path
import shutil

#@markdown ---
#@markdown ### Modello Demucs
modello = "htdemucs" #@param ["htdemucs", "htdemucs_6s", "htdemucs_ft"]
#@markdown - `htdemucs`: standard (drums, bass, vocals, other)
#@markdown - `htdemucs_6s`: 6 stems (+ guitar, piano)
#@markdown - `htdemucs_ft`: fine-tuned, migliore qualita

print("📤 Carica il file audio da separare...")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    audio_path = Path(f"/content/cantautore/temp/{filename}")
    shutil.move(filename, str(audio_path))

    from export_kit import separa_stems

    output_dir = Path(f"/content/cantautore/output/{audio_path.stem}_stems")
    stems = separa_stems(audio_path, output_dir, model=modello)

    print(f"\n✅ Stems separati in: {output_dir}")
    print(f"\n🎧 Ascolta ogni stem:")

    import IPython.display as ipd
    for name, path in sorted(stems.items()):
        p = Path(path)
        if p.exists():
            print(f"\n{'─'*40}")
            print(f"🎵 {name}")
            ipd.display(ipd.Audio(str(p)))
else:
    print("❌ Nessun file caricato")

In [ ]:
#@title ℹ️ **Info canzoni generate**
#@markdown Mostra tutte le canzoni e i kit disponibili

from pathlib import Path
import json

output_dir = Path("/content/cantautore/output")

wavs = sorted(output_dir.glob("*.wav"), key=lambda f: f.stat().st_mtime, reverse=True)
wavs = [w for w in wavs if "_kit" not in str(w.parent.name)]

kits = sorted(output_dir.glob("*_kit"), key=lambda f: f.stat().st_mtime, reverse=True)
kits = [k for k in kits if k.is_dir()]

print(f"📊 STATO STUDIO")
print(f"{'='*50}")
print(f"🎵 Canzoni: {len(wavs)}")
print(f"📦 Kit stems: {len(kits)}")

total_size = sum(f.stat().st_size for f in output_dir.rglob("*") if f.is_file())
print(f"💾 Spazio totale: {total_size / (1024*1024):.0f} MB")

# Voci disponibili
voices_dir = Path("/content/cantautore/voices")
voices = [d.name for d in voices_dir.iterdir() if d.is_dir()]
print(f"🎤 Voci: {', '.join(voices) if voices else 'nessuna (usa voce Lyria)'}")

print(f"\n{'='*50}")
print("🎵 CANZONI:")
for i, wav in enumerate(wavs):
    import soundfile as sf
    info = sf.info(str(wav))
    dur = f"{int(info.duration // 60)}:{int(info.duration % 60):02d}"
    size = wav.stat().st_size / (1024*1024)

    testo_f = wav.parent / f"{wav.stem}_testo.json"
    title = wav.stem
    if testo_f.exists():
        d = json.loads(testo_f.read_text())
        title = d.get("titolo", wav.stem)

    kit_exists = (wav.parent / f"{wav.stem}_kit").exists()
    kit_label = " 📦" if kit_exists else ""

    print(f"  {i+1}. {title} [{dur}] {size:.1f}MB{kit_label}")

if kits:
    print(f"\n{'='*50}")
    print("📦 KIT STEMS:")
    for kit in kits:
        stems = list(kit.glob("*.wav"))
        print(f"  {kit.name}: {len(stems)} file")
        for s in sorted(stems):
            print(f"    - {s.name} ({s.stat().st_size / (1024*1024):.1f}MB)")